In [2]:
import os
import sys
import json

from pathlib import Path
import pandas as pd

path_proj = Path.cwd().parent
path_data = os.path.join(path_proj, "data")

sys.path.append(os.path.join(path_proj, "utils"))

from generate_llm import generate_gpt, get_embeddings


%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
prompt = """
<persona>
You are an economist specialized in analyzing central bank communications and monetary policy signals.
</persona>

<context>
Federal Reserve publication.

Title:
{title}

Publication date:
{publication_date}

Text:
{text}
</context>

<task>

Extract monetary policy signals from the publication and classify each signal according to the monetary policy stance it implies.

- dovish: Clearly supports monetary easing or maintaining an accommodative stance. Use when the signal indicates that inflation is at target, below target, or moving sustainably toward target; economic activity is weakening; or labor market conditions are softening in a way that reduces the need for restrictive policy.
- mostly_dovish: Leans toward monetary easing or delaying further tightening, but it is not definitive. Use when the signal reports progress on inflation, slower growth, or softer labor market conditions, while also mentioning uncertainty.
- neutral: The signal does not clearly imply either easing or tightening. Use when it does not indicate a dominant direction for policy. Do not classify as neutral merely because it is factual or descriptive.
- mostly_hawkish: Leans toward monetary tightening or delaying easing, but it is not definitive. Use when the signal reports elevated inflation, upside inflation risks, resilient demand, or strong labor market conditions, while also mentioning uncertainty.
- hawkish: Clearly supports monetary tightening or maintaining a restrictive stance. Use when the signal indicates that inflation is above target, persistent, broad-based, or rising; economic activity remains strong; or labor market conditions remain tight in a way that justifies restrictive policy.

Return a JSON object with the following structure:
{
  "dovish": [
    {
      "signal": "monetary policy signal",
      "justification": "brief explanation of why this signal is classified as dovish",
      "evidence": "short supporting excerpt from the text"
    }
  ],
  "mostly_dovish": [
    {
      "signal": "monetary policy signal",
      "justification": "brief explanation of why this signal is classified as mostly_dovish",
      "evidence": "short supporting excerpt from the text"
    }
  ],
  "neutral": [
    {
      "signal": "monetary policy signal",
      "justification": "brief explanation of why this signal is classified as neutral",
      "evidence": "short supporting excerpt from the text"
    }
  ],
  "mostly_hawkish": [
    {
      "signal": "monetary policy signal",
      "justification": "brief explanation of why this signal is classified as mostly_hawkish",
      "evidence": "short supporting excerpt from the text"
    }
  ],
  "hawkish": [
    {
      "signal": "monetary policy signal",
      "justification": "brief explanation of why this signal is classified as hawkish",
      "evidence": "short supporting excerpt from the text"
    }
  ]
}
</task>

<guidelines>
- Do not extract signals unrelated to monetary policy.
- Each signal should express one main idea.
- Evidence must be a short excerpt copied from the text.
- If a category has no relevant signals, return an empty list.
</guidelines>

"""

In [4]:
score_map = {
    "dovish": -1,
    "mostly_dovish": -0.5,
    "neutral": 0,
    "mostly_hawkish": 0.5,
    "hawkish": 1
}

In [5]:
links_text_path = os.path.join(path_data, "links_text.txt")
result_analysis_path = os.path.join(path_data, "result_analysis.csv")

expected_columns = [
    "url",
    "title",
    "publication_date",
    "classification",
    "score",
    "signal",
    "justification",
    "evidence",
    "embedding_signal",
]

expected_classes = [
    "dovish",
    "mostly_dovish",
    "neutral",
    "mostly_hawkish",
    "hawkish",
]

with open(links_text_path, "r", encoding="utf-8") as f:
    links_text = json.load(f)

try:
    df_result_all = pd.read_csv(result_analysis_path)
except FileNotFoundError:
    df_result_all = pd.DataFrame(columns=expected_columns)

number_items_to_process = 9999999
processed_count = 0

for source_item in links_text:
    url = source_item["url"]
    title = source_item["title"]
    text = source_item["text"]
    publication_date = source_item["publication_date"]

    if "NOT EXTRACTED - IMPLEMENTATION NOTE" in title:
        print(f"Skipping URL with implementation note: {url}")
        continue

    if url in df_result_all["url"].values:
        print(f"Skipping already processed URL: {url}")
        continue

    if not text or text.strip() == "":
        print(f"Skipping empty text for URL: {url}")
        continue

    processed_count += 1
    if processed_count > number_items_to_process:
        break

    print(f"Processing: {title}")
    print(f"Publication Date: {publication_date}")

    prompt_filled = (
        prompt.replace("{title}", title)
        .replace("{text}", text)
        .replace("{publication_date}", publication_date)
    )

    llm_parameters = {
        "model": "gpt-5.6-terra",
        "json_mode": True,
        "temperature": 0,
        "reasoning_effort": "none"
    }

    response = generate_gpt(prompt_filled, llm_parameters)
    print(f"Response: {response}")

    try:
        response_json = json.loads(response)
    except json.JSONDecodeError as exc:
        print(f"Invalid JSON for URL {url}: {exc}")
        print("Response preview:")
        print(repr(response[:1000]))
        continue

    if not isinstance(response_json, dict):
        print(f"Unexpected response type for URL {url}: {type(response_json).__name__}")
        continue

    rows = []

    for classification in expected_classes:
        data = response_json.get(classification, [])

        print(f"Classification: {classification}")
        print(f"Data: {data}")

        if not data:
            continue

        if not isinstance(data, list):
            print(f"Skipping invalid payload for {classification}: expected list")
            continue

        for extracted_item in data:
            if not isinstance(extracted_item, dict):
                print(f"Skipping invalid entry in {classification}: {extracted_item}")
                continue

            signal = extracted_item.get("signal", "").strip()
            justification = extracted_item.get("justification", "").strip()
            evidence = extracted_item.get("evidence", "").strip()

            if not signal:
                print(f"Skipping empty key point in {classification}")
                continue

            print(f"Signal: {signal}")
            print(f"Evidence: {evidence}")

            try:
                embedding = get_embeddings(signal)
            except Exception as exc:
                print(f"Embedding failed for URL {url}: {exc}")
                continue

            rows.append(
                {
                    "url": url,
                    "title": title,
                    "publication_date": publication_date,
                    "classification": classification,
                    "score": score_map[classification],
                    "signal": signal,
                    "justification": justification,
                    "evidence": evidence,
                    "embedding_signal": embedding,
                }
            )

    if not rows:
        print(f"No valid rows generated for URL: {url}")
        continue

    df_result = pd.DataFrame(rows, columns=expected_columns)
    df_result_all = pd.concat([df_result_all, df_result], ignore_index=True)
    df_result_all.to_csv(result_analysis_path, index=False)

Skipping already processed URL: https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128a.htm
Skipping already processed URL: https://www.federalreserve.gov/monetarypolicy/fomcpressconf20260128.htm
Skipping already processed URL: https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128b.htm
Processing: Implementation Note issued January 28, 2026
Publication Date: 2026-01-28
Response(id='resp_0223db0bfbff0dd6006aab265f62a887d28963000bf5080dcd', created_at=1789601375.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-terra', object='response', output=[ResponseOutputMessage(id='msg_0223db0bfbff0dd6006aab2660172087d28029a75113900e72', content=[ResponseOutputText(annotations=[], text='{\n  "dovish": [],\n  "mostly_dovish": [],\n  "neutral": [],\n  "mostly_hawkish": [],\n  "hawkish": []\n}', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=Tr